# SV model with normal-mixture approximation

本 Notebook 演示如何在本地数据文件夹中加载分钟级期货数据，使用混合正态近似的随机波动（SV）模型进行快速 MCMC 演示。所有注释使用中文，print 输出和图表元素使用英文，便于跨平台显示。

In [ ]:
# 导入依赖，注释使用中文
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# 可视化与进度显示
import matplotlib.pyplot as plt
import seaborn as sns

# 自定义模块
from sv_toolkit.data import list_csv_files, load_contracts_in_dir, get_contract_symbol_from_path
from sv_toolkit.mcmc import run_mcmc_sv
from sv_toolkit.batch import make_timestamped_root, save_param_summary
from sv_toolkit.plotting import (
    plot_returns,
    plot_volatility,
    plot_return_histogram,
    plot_acf_returns,
    plot_intraday_pattern,
    plot_param_posterior,
    plot_vol_and_abs_returns,
    plot_standardized_residuals,
    plot_mixture_usage,
)

# 设置 matplotlib 显示风格
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# 确定数据目录，默认为当前仓库下的 2005年__20250905 文件夹
data_dir = Path('../2005年__20250905')

# 创建带时间戳的输出目录，方便保存图片与参数（每个合约会在下级目录中落盘）
output_root = make_timestamped_root(Path('outputs'))
print(f'Environment ready. Data dir: {data_dir}. Output root: {output_root}')


## 1. 浏览数据文件



In [ ]:
# 列出数据目录中的前 5 个文件，避免一次性打印全部
preview_files = list_csv_files(data_dir, max_files=5)
print('Preview finished.')

## 2. 读取单个/少量文件并构造收益率



In [ ]:
# 选择数据子集，按合约独立加载，便于快速跑通流程
datasets = load_contracts_in_dir(
    data_dir,
    contract_code=None,  # 可以填入具体合约代码，比如 'A0505.XDCE'
    start_time=None,     # 可以填入 '2005-01-01' 这样的字符串
    end_time=None,       # 可以填入 '2008-12-31' 等字符串
    max_files=5,         # 默认尝试读取前 5 个文件
    max_rows_per_file=10000,  # 限制行数，方便快速运行
)
symbols = list(datasets.keys())
if not symbols:
    raise RuntimeError('No contracts loaded from the provided directory.')
print(f'Successfully loaded {len(symbols)} contracts: {symbols}')


## 3. 运行简化版 MCMC



In [ ]:
# 针对每个合约独立运行 MCMC、保存参数并绘图（全部输出到各自文件夹）
results_dict = {}
base_seed = 2025

for idx, contract_tag in enumerate(symbols):
    print("
" + "=" * 60)
    print(f'Processing contract: {contract_tag}')

    contract_data = datasets[contract_tag]
    r = contract_data['r']
    y_star = contract_data['y_star']
    df = contract_data['df']
    exog = contract_data.get('exog')

    contract_dir = output_root / contract_tag
    contract_dir.mkdir(parents=True, exist_ok=True)

    print(f'Contract {contract_tag}: returns length = {len(r)}, dataframe rows = {len(df)}')

    mcmc_results = run_mcmc_sv(
        r=r,
        y_star=y_star,
        exog_state=exog,
        n_iter=240,      # 演示用较小迭代次数
        burn_in=40,
        thin=2,
        rng_seed=base_seed + idx,  # 不同合约使用不同种子
        progress_every=20,
    )

    results_dict[contract_tag] = {'mcmc_results': mcmc_results, 'r': r, 'y_star': y_star, 'df': df}

    extra_info = {
        "T": len(r),
        "n_iter": 240,
        "burn_in": 40,
        "thin": 2,
        "data_dir": str(data_dir),
        "contract_tag": contract_tag,
    }
    save_param_summary(mcmc_results, contract_dir, contract_tag, extra_info=extra_info)
    print(f'Parameter summary saved under: {contract_dir}')

    # 绘制并保存所有诊断图，文件名保留原有格式
    returns_png = plot_returns(df, contract_dir, title_suffix=contract_tag)
    vol_png = None
    if len(mcmc_results['h']) > 0:
        vol_png = plot_volatility(mcmc_results['h'], df, contract_dir, title_suffix=contract_tag)

    hist_png = plot_return_histogram(r, contract_dir, title_suffix=contract_tag)
    acf_png = plot_acf_returns(r, contract_dir, title_suffix=contract_tag)
    intra_png = plot_intraday_pattern(df, contract_dir, title_suffix=contract_tag)
    param_png = plot_param_posterior(mcmc_results, contract_dir, title_suffix=contract_tag)
    vol_abs_png = plot_vol_and_abs_returns(mcmc_results["h"], df, contract_dir, title_suffix=contract_tag)
    std_resid_png = plot_standardized_residuals(r, mcmc_results, contract_dir, title_suffix=contract_tag)
    mix_png = None
    if 's' in mcmc_results and len(mcmc_results['s']) > 0:
        mix_png = plot_mixture_usage(mcmc_results['s'], contract_dir, title_suffix=contract_tag)

    print('Figures saved:', returns_png, vol_png, hist_png, acf_png, intra_png, param_png, vol_abs_png, std_resid_png, mix_png)
    print(f'✓ Finished contract {contract_tag}')

print("
" + "=" * 60)
print(f'Summary: processed {len(results_dict)} of {len(symbols)} contracts')
print(f'Outputs located under: {output_root}')


## 4. 保存参数摘要到时间戳目录



In [ ]:
# 参数保存和绘图已在上一单元完成，无需额外代码。


## 5. 绘制收益率、隐含波动率与其他诊断图



In [ ]:
# 上述循环已完成所有合约的输出。可以在此处添加自定义汇总。
